# 14 · Contrastive image-text learning: CLIP and SigLIP losses

**Needed for:** CLIP and SigLIP (vision track, step V2). SigLIP encoders are the default vision tower in most
current VLMs (LLaVA-OneVision, PaliGemma, Janus uses SigLIP for understanding).

Train an image encoder and a text encoder so matching pairs have high cosine similarity and non-matching pairs
low. With a batch of $B$ pairs, the $B\times B$ similarity matrix has the positives on its diagonal.

- **CLIP:** softmax cross-entropy over each row (image → which text?) and each column (text → which image?).
  Each row's softmax needs the *whole* row, so in distributed training every GPU all-gathers all embeddings.
- **SigLIP:** treat every one of the $B^2$ pairs as an independent binary classification (match or not) with a
  sigmoid. No normalization across the batch, so it can be computed in chunks and works well at small batch sizes.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. CLIP loss

$$\text{logits} = e^{\tau}\cdot \hat I \hat T^\top, \qquad \mathcal{L} = \tfrac12\big[\text{CE}(\text{logits}, \text{diag}) + \text{CE}(\text{logits}^\top, \text{diag})\big]$$

$\hat I, \hat T$ are L2-normalized. The temperature is learned as $\log$ of the scale, initialized to
$\log(1/0.07)$ and clamped at 100 so training doesn't become unstable.

In [ ]:
def clip_loss(img_emb, txt_emb, logit_scale):
    img, txt = F.normalize(img_emb, dim=-1), F.normalize(txt_emb, dim=-1)
    logits = logit_scale.exp().clamp(max=100) * img @ txt.T
    labels = torch.arange(len(img), device=img.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2


def clip_loss_loop(img_emb, txt_emb, logit_scale):
    """Slow reference: one row and one column at a time."""
    img, txt = F.normalize(img_emb, dim=-1), F.normalize(txt_emb, dim=-1)
    s = logit_scale.exp().clamp(max=100)
    B, total = len(img), 0.0
    for i in range(B):
        row = s * (txt @ img[i])                      # image i vs every text
        col = s * (img @ txt[i])                      # text i vs every image
        total += -(row[i] - row.logsumexp(0)) - (col[i] - col.logsumexp(0))
    return total / (2 * B)


def test_clip_loss(fn):
    torch.manual_seed(0)
    img, txt, scale = torch.randn(8, 16), torch.randn(8, 16), torch.tensor(math.log(1 / 0.07))
    torch.testing.assert_close(fn(img, txt, scale), clip_loss_loop(img, txt, scale))
    assert fn(txt, txt, scale) < fn(img, txt, scale)          # perfect pairs give lower loss


test_clip_loss(clip_loss)
print("CLIP loss matches the loop reference")

## 2. SigLIP loss

$$\mathcal{L} = -\frac{1}{B}\sum_{i}\sum_{j} \log \sigma\big(z_{ij}\,(t\,\hat x_i\cdot\hat y_j + b)\big), \qquad z_{ij} = \begin{cases}1 & i=j\\-1 & i\ne j\end{cases}$$

Learned temperature $t$ (init $\log t = \log 10$) and **bias** $b$ (init $-10$). The bias matters: with $B$
pairs there are $B^2 - B$ negatives, so at init the model should predict "not a match" for nearly everything.

In [ ]:
def siglip_loss(img_emb, txt_emb, log_t, b):
    img, txt = F.normalize(img_emb, dim=-1), F.normalize(txt_emb, dim=-1)
    logits = log_t.exp() * img @ txt.T + b
    z = 2 * torch.eye(len(img), device=img.device) - 1          # +1 diagonal, -1 elsewhere
    return -F.logsigmoid(z * logits).sum() / len(img)


def test_siglip_loss(fn):
    torch.manual_seed(0)
    img, txt = torch.randn(6, 16), torch.randn(6, 16)
    log_t, b = torch.tensor(math.log(10.0)), torch.tensor(-10.0)
    i_n, t_n = F.normalize(img, dim=-1), F.normalize(txt, dim=-1)
    ref = 0.0
    for i in range(6):
        for j in range(6):
            z = 1.0 if i == j else -1.0
            ref += -F.logsigmoid(z * (10.0 * (i_n[i] @ t_n[j]) - 10.0))
    torch.testing.assert_close(fn(img, txt, log_t, b), ref / 6)


test_siglip_loss(siglip_loss)
print("SigLIP loss matches the pairwise reference")

## 3. Train two encoders on synthetic paired data

Each "concept" is a hidden vector. The "image" and the "caption" of a concept are two different random nonlinear
views of it plus noise. The encoders must discover that the two views share information. Measure retrieval:
for each image, is its own caption the most similar one among 256 candidates?

In [ ]:
LATENT, D_IMG, D_TXT, EMB = 16, 64, 48, 32
torch.manual_seed(0)
view_img = nn.Sequential(nn.Linear(LATENT, D_IMG), nn.Tanh(), nn.Linear(D_IMG, D_IMG)).requires_grad_(False)
view_txt = nn.Sequential(nn.Linear(LATENT, D_TXT), nn.Tanh(), nn.Linear(D_TXT, D_TXT)).requires_grad_(False)


def paired_batch(B):
    z = torch.randn(B, LATENT)
    return view_img(z) + 0.25 * torch.randn(B, D_IMG), view_txt(z) + 0.25 * torch.randn(B, D_TXT)


def retrieval_acc(enc_i, enc_t, B=256):
    with torch.no_grad():
        x, y = paired_batch(B)
        sim = F.normalize(enc_i(x), dim=-1) @ F.normalize(enc_t(y), dim=-1).T
        return (sim.argmax(1) == torch.arange(B)).float().mean().item()


def train(loss_name, steps=600, B=128):
    torch.manual_seed(1)
    enc_i = nn.Sequential(nn.Linear(D_IMG, 128), nn.GELU(), nn.Linear(128, EMB))
    enc_t = nn.Sequential(nn.Linear(D_TXT, 128), nn.GELU(), nn.Linear(128, EMB))
    scale = nn.Parameter(torch.tensor(math.log(1 / 0.07) if loss_name == "clip" else math.log(10.0)))
    bias = nn.Parameter(torch.tensor(-10.0))
    opt = torch.optim.AdamW([*enc_i.parameters(), *enc_t.parameters(), scale, bias], lr=1e-3)
    print(f"{loss_name}: retrieval@1 before training {retrieval_acc(enc_i, enc_t):.2f}", end="")
    for _ in range(steps):
        x, y = paired_batch(B)
        loss = clip_loss(enc_i(x), enc_t(y), scale) if loss_name == "clip" else siglip_loss(enc_i(x), enc_t(y), scale, bias)
        opt.zero_grad()
        loss.backward()
        opt.step()
    print(f"  after {retrieval_acc(enc_i, enc_t):.2f}  (chance 1/256 = {1 / 256:.3f})")


train("clip")
train("siglip")

## 4. Zero-shot classification

Embed one text prompt per class ("a photo of a {class}"), embed the image, pick the class with the highest
cosine similarity. No classifier is trained: the class list can change at test time. Prompt ensembling
(averaging embeddings over several templates) typically adds a few points of accuracy.

## Things to be able to explain

- Why does CLIP need huge batches (32k), and why does SigLIP not? (Softmax needs many negatives per row to be
  informative; the sigmoid loss treats each pair independently.)
- How do you compute the CLIP loss across 64 GPUs? (All-gather embeddings, not images; each GPU computes its rows.
  Gradients need care: the all-gather must be differentiable, or use the local-loss trick.)
- Why L2-normalize before the dot product? (Cosine similarity bounds logits; the temperature sets their scale.)

## Drill

In [ ]:
def my_clip_loss(img_emb, txt_emb, logit_scale):
    raise NotImplementedError


def my_siglip_loss(img_emb, txt_emb, log_t, b):
    raise NotImplementedError


drill(test_clip_loss, my_clip_loss)
drill(test_siglip_loss, my_siglip_loss)